# C10-competition-craft — Practice p10 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()


def run_experiment():
    rng = np.random.default_rng(SEED)
    perm = rng.permutation(len(df))
    val_idx, tr_idx = perm[:150], perm[150:]
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=7)),
    ])
    pipe.fit(X.iloc[tr_idx], y[tr_idx])
    preds = pipe.predict(X.iloc[val_idx])
    return f1_score(y[val_idx], preds, average="macro"), preds


f1_a, preds_a = run_experiment()
f1_b, preds_b = run_experiment()
deterministic = bool(f1_a == f1_b and (preds_a == preds_b).all())
(f1_a, f1_b, deterministic)

The original breaks D1 by drawing its permutation from an unseeded legacy global generator, which also prevents D4's deterministic re-run. The audit uses exact equality because identical code, data, environment, and seeds have no legitimate source of even last-bit variation.

### Answer check

In [ ]:
assert np.isclose(f1_a, 0.7638515057640547, atol=1e-12, rtol=0)
assert f1_a == f1_b
assert np.array_equal(preds_a, preds_b)
assert deterministic is True
assert preds_a.shape == (150,)